In [1]:
#this notebook made in reference to 5 days ai agents intensive course with Google

In [2]:
#Istalling dependencies
!pip install google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 4.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich

In [3]:
#authenticate the key
import os
from kaggle_secrets import UserSecretsClient
try:
    UserSecretsClient=UserSecretsClient()#get_secret is a instance method so we created a instance here
    GEMINI_API_KEY=UserSecretsClient.get_secret("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"]=GEMINI_API_KEY
    os.environ["GEMINI_GENAI_USE_VERTEXAI"]="FALSE"#this tells which backend to use for ADK
    print("Gemini api setup complete")
except Exception as e:
    print(f"add api key : details {e}")

Gemini api setup complete


In [4]:
#import components from ADK
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types
print("imports successful")

imports successful


In [5]:
#retry options config
retry_config=types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504]
)

In [6]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [7]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


In [8]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [9]:
#running agent
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug("what are latest technologies of AI?")


 ### Created new session: debug_session_id

User > what are latest technologies of AI?


ResearchCoordinator > The latest technologies in AI are rapidly evolving. Key advancements include:

*   **Generative AI and Multimodality:** Large Language Models (LLMs) such as ChatGPT, Gemini, and Claude are increasingly sophisticated, exhibiting human-like conversational skills and the ability to generate various forms of content including text, images, audio, and video. Future developments aim to improve their reasoning capabilities for more autonomous task execution.
*   **Embodied AI and Robotics:** AI is extending into the physical realm through embodied AI and advanced robotics. This involves generative AI powering humanoid robots capable of performing complex tasks with precision, thus integrating AI into physical operations.
*   **Edge AI and Ethical Considerations:** Edge AI allows AI models to run directly on devices, enabling real-time processing and enhancing data privacy. Alongside this, Explainable AI (XAI) and ethical AI practices are gaining importance to ensure tran

In [10]:
#we cannot rely on previous multi agent because it based on instructions of prompt, this is unreliable because complex LLM might skip the given  order.
#let us build apipeline to solve this problem
#1. linear pipeline-SequentialAent-This is perfect for tasks that build on each other, but it's slow if the tasks are independent.
# Outline Agent: Creates the initial blog post outline.
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  # The result of this agent will be stored in the session state with this key.
)

print(" outline_agent created.")

 outline_agent created.


In [11]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  # The result of this agent will be stored with this key.
)

print(" writer_agent created.")

 writer_agent created.


In [12]:
# Editor Agent: Edits and polishes the draft from the writer agent.
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print(" editor_agent created.")

 editor_agent created.


In [13]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print(" Sequential Agent created.")

 Sequential Agent created.


In [14]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of multi-agent systems for software developers"
)


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of multi-agent systems for software developers
OutlineAgent > ## Multi-Agent Systems: Your New Development Superpowers 🚀

Ever felt like your software projects are wrestling with complexities that just won't quit? Imagine a world where your code doesn't just execute, but intelligently collaborates. That's the promise of multi-agent systems (MAS), and for software developers, it's a game-changer.

### **1. Decomposing Complexity into Collaborating Units**

*   **Modular Miracles:** Break down dauntingly large systems into smaller, manageable, and independent agents. This makes development, testing, and debugging significantly less painful.
*   **Specialized Expertise:** Each agent can be designed with a specific role and set of skills, leading to more efficient and targeted solutions for different aspects of your software.
*   **Easier Maintenance & Evolution:** Updating or replacing a single agent 

In [15]:
#Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently
# Tech Researcher: Focuses on AI and ML trends.
tech_researcher = Agent(
    name="TechResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (100 words).""",
    tools=[google_search],
    output_key="tech_research",  # The result of this agent will be stored in the session state with this key.
)

print(" tech_researcher created.")

 tech_researcher created.


In [16]:
# Health Researcher: Focuses on medical breakthroughs.
health_researcher = Agent(
    name="HealthResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research recent medical breakthroughs. Include 3 significant advances,
their practical applications, and estimated timelines. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="health_research",  # The result will be stored with this key.
)

print(" health_researcher created.")

 health_researcher created.


In [17]:
# Finance Researcher: Focuses on fintech trends.
finance_researcher = Agent(
    name="FinanceResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="finance_research",  # The result will be stored with this key.
)

print(" finance_researcher created.")

 finance_researcher created.


In [18]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}
    
    **Health Breakthroughs:**
    {health_research}
    
    **Finance Innovations:**
    {finance_research}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 200 words.""",
    output_key="executive_summary",  # This will be the final output of the entire system.
)

print(" aggregator_agent created.")

 aggregator_agent created.


In [19]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print(" Parallel and Sequential Agents created.")

 Parallel and Sequential Agents created.


In [20]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
TechResearcher > Here are three key AI/ML trends, the companies involved, and their potential impact:

1.  **Generative AI and Complex Content Creation:** This trend involves AI models capable of generating diverse content, including text, images, video, and music. Companies like **OpenAI** (GPT models), **Google** (Imagen, Muse), and **Meta** are heavily involved. The impact is transformative for artistic expression and practical applications, potentially expanding creativity and automating content production across industries.

2.  **Shift from Large Language Models (LLMs) to Small Language Models (SLMs):** There's a move towards more efficient and specialized smaller models. While specific companies leading this shift are less publicized, major AI players are likely researching and developing SLMs for targeted applications. The impact lies in enabling AI on devices with 

In [21]:
#Loop when: Iterative improvement is needed, quality refinement matters, or you need repeated cycles.
# This agent runs ONCE at the beginning to create the first draft.
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the user's prompt, write the first draft of a short story (around 100-150 words).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story",  # Stores the first draft in the state.
)

print(" initial_writer_agent created.")

 initial_writer_agent created.


In [22]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print(" critic_agent created.")

 critic_agent created.


In [23]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print(" exit_loop function created.")

 exit_loop function created.


In [24]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print(" refiner_agent created.")

 refiner_agent created.


In [25]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print(" Loop and Sequential Agents created.")

 Loop and Sequential Agents created.


In [26]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a lighthouse keeper who discovers a mysterious, glowing map"
)


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > Elias traced the salt-crusted rim of his lantern, the familiar weight a comfort against the gnawing loneliness. For twenty years, the lighthouse had been his world, a stoic sentinel against the churning sea. One blustery evening, during his routine sweep of the storeroom, his hand brushed against a loose floorboard. Beneath it, nestled in damp sand, lay a rolled-up parchment. Unfurling it, Elias gasped. It wasn't paper, but some impossibly smooth, cool material that shimmered with an inner light. Strange symbols, like constellations he'd never seen, pulsed and shifted across its surface. A map, certainly, but of what? The glowing lines seemed to beckon, hinting at journeys far beyond his isolated rock.
CriticAgent > Here are a few suggestions to enhance your story:

1.  **Show, Don't Just Tell Elias's Loneliness:** While you esta